List the number of comments on closed 100 PRs by creation order.

In [ ]:
token = ''

import requests
import json
import pandas as pd

# Get Comments

url = f"https://api.github.com/graphql"
headers = {
        "Authorization": f"Bearer {token}"
    }
query = """{
    repository(owner: "arc53", name: "DocsGPT") {
        pullRequests(states: CLOSED, last: 100) {
        nodes {
            number
            createdAt
            # Comments in the traditional sense
            commentThreads: comments(first: 50) {
                totalCount
                nodes { author { login } }
            }
            # Inline code comments.
            reviewThreads(last: 50) {
                totalCount
                nodes { 
                    comments(first: 50) {
                        nodes {
                            author { login }
                        }
                    }
                }
            }
        }
    }    
}
}
"""

response = requests.post(url, json={'query': query}, headers=headers)
# # Dump json for reference
# with open("bot_comments.json", "w") as file:
#     json.dump(response.json(), file, indent=4)


In [97]:
df = pd.DataFrame(response.json()["data"]["repository"]["pullRequests"]["nodes"])
# Ensure created at is a date and sort 
df['createdAt'] = pd.to_datetime(df['createdAt'])
df_sorted = df.sort_values(by='createdAt', ascending=False)

def check_comment_thread_is_bot(comment_obj):
    for node in comment_obj['nodes']:
        if 'bot' in node['author']['login'].lower():
            return True

    return False

def check_review_thread_is_bot(comment_obj):
    if not comment_obj['nodes']:
        return False
    
    # Specific line of code
    for comment_node in comment_obj['nodes']:
        # Comments
        inner_nodes = comment_node['comments']['nodes']
        # Get author username
        for key in inner_nodes:
            if 'bot' in key['author']['login'].lower():
                return True

    return False

# Add columns to determine if the PR has "at least one comment was added by a bot"
df_sorted['commentThreatHasBot'] = df_sorted['commentThreads'].apply(check_comment_thread_is_bot)
df_sorted['reviewThreatHasBot'] = df_sorted['reviewThreads'].apply(check_review_thread_is_bot)

df_sorted['prHasBot'] = df_sorted['commentThreatHasBot'] | df_sorted['reviewThreatHasBot']

df_sorted['totalComments'] = df_sorted['commentThreads'].str.get('totalCount') + df_sorted['reviewThreads'].str.get('totalCount')

df_sorted.head(5)


,number,createdAt,commentThreads,reviewThreads,commentThreatHasBot,reviewThreatHasBot,prHasBot,totalComments
99,2271,2026-02-03 17:13:50+00:00,"{'totalCount': 1, 'nodes': [{'author': {'login...","{'totalCount': 0, 'nodes': []}",False,False,False,1
98,2251,2026-01-08 06:40:15+00:00,"{'totalCount': 2, 'nodes': [{'author': {'login...","{'totalCount': 0, 'nodes': []}",True,False,True,2
97,2248,2026-01-04 11:19:23+00:00,"{'totalCount': 2, 'nodes': [{'author': {'login...","{'totalCount': 0, 'nodes': []}",False,False,False,2
96,2243,2025-12-29 20:06:13+00:00,"{'totalCount': 2, 'nodes': [{'author': {'login...","{'totalCount': 0, 'nodes': []}",True,False,True,2
95,2242,2025-12-29 20:06:10+00:00,"{'totalCount': 2, 'nodes': [{'author': {'login...","{'totalCount': 0, 'nodes': []}",True,False,True,2


Create CSV File

In [98]:
final_df = df_sorted[['number', 'totalComments']]
final_df = final_df.rename(columns={'number': 'PR Number', 'totalComments': 'Number of Comments'})
final_df.to_csv("pr_and_number_of_comments.csv", index=False)

Compute the average number of comments from PRs where at least one comment was added by a bot and where no bot was involved.

In [99]:
bots_count = df_sorted['prHasBot'].sum()
humans_count = 100 - bots_count

comments_with_at_least_one_bot = df_sorted.loc[df_sorted['prHasBot'] == True, 'totalComments'].sum()
comments_with_no_bots = df_sorted.loc[df_sorted['prHasBot'] == False, 'totalComments'].sum()

print("AVG # of comments from PRs where at least one comment was added by a bot:", comments_with_at_least_one_bot / bots_count)
print("AVG # of comments from PRs where no bot was involved:", comments_with_no_bots / humans_count)

AVG # of comments from PRs where at least one comment was added by a bot: 2.017857142857143
AVG # of comments from PRs where no bot was involved: 2.659090909090909


Interestingly, PRs with 'bot' have less comments than PRs with no 'bot'.

This is mainly due to two outlier human PRs (10 and 13 comments each) and how bots are formally classified in this assignment.

